# Super Filter v3: Robust Street View invalid-image pipeline

This notebook replaces the brittle single-rule filtering approach with a layered multi-cue scoring system focused on low false positives for building-visible images.

What this version adds:
- central, tunable config for all thresholds and weights
- modular detectors (darkness, glare, blur, building/facade, open-scene, corridor-view, interior, occlusion, placeholder similarity, non-Google)
- hard-reject + soft-combined decision logic with rescue logic for building-visible borderline frames
- resumable processing for large runs (~20k images)
- CSV/JSON/JSONL outputs with per-image diagnostics and confidence
- invalid image routing into reason folders
- debug thumbnails and review sample manifests
- lightweight synthetic self-tests for sanity-checking core detectors


## STEP 1 - Existing pipeline review and failure diagnosis

### Current architecture in prior notebook
- Single-script Colab notebook with mostly hardcoded thresholds and per-image flags.
- Heavy reliance on DeepLab semantic segmentation for `building_ratio` with a single `building_ratio < 0.15` rule.
- Global blur via Laplacian variance only (`blur_score < 100`), no region awareness.
- Template match for Google logo in bottom-left without scale safety (caused OpenCV assertion crash when template > crop).
- Placeholder/no-imagery check using one average hash tolerance (`abs(hash - ref_hash) < 5`) only.
- CLIP zero-shot day/night and indoor/outdoor used as binary gate, not integrated with other cues.
- Any triggered detector copies image into category folder; no robust combined scoring policy.

### Why prior filters failed or overfired
1. **No-buildings overfired**: depended on segmentation ratio threshold and ignored structural facade cues; segmentation miss => false positives.
2. **Sky/trees/vehicle filters missed**: they were mostly removed or not integrated meaningfully in final logic.
3. **Blur missed many blurry images**: whole-image Laplacian variance is unstable when sky/flat regions dominate; no facade ROI.
4. **Street-corridor vs facade not handled**: no dedicated corridor score combining road center dominance + vanishing geometry + low side-facade presence.
5. **No-imagery exactness issue**: single hash tolerance can miss near-identical placeholders and has weak diagnostics.
6. **Interior/no-google noisy**: brittle logo/template assumptions and binary CLIP labels, with little outdoor evidence cross-check.
7. **Operational fragility**: no resumable manifest logic, minimal diagnostics, no confidence decomposition, and crash on logo template sizing.

### What is retained
- Bright/glare guardrail remains strong.
- Dark filter retained but tuned to prioritize clear-night cases.
- Placeholder-reference strategy retained but redesigned as multi-similarity (pHash/dHash/correlation).


## STEP 2 - Redesign plan before implementation

### New design
1. **Feature extraction layer (lightweight, OpenCV/NumPy)**
   - brightness/darkness, glare saturation/clipping, blur (ROI Laplacian + Tenengrad + edge density), building/facade structure, sky/vegetation/road masks, road-corridor geometry, occlusion, interior likelihood, placeholder similarity, non-google signal.
2. **Decision layer**
   - hard rejects for extreme unusable conditions (placeholder, extreme glare, extreme darkness, severe blur+low facade, strong interior).
   - weighted soft invalid score + synergy bonuses for combinations (low building + open scene, low building + corridor, blur + low facade, occlusion + low facade).
   - rescue logic to keep borderline frames when building evidence is strong (reduces false positives).
3. **Output/diagnostics layer**
   - per-image full score row with thresholds and contribution terms.
   - CSV + JSON + JSONL manifests.
   - optional invalid image copy/move into primary-reason subfolders.
   - debug overlays and review sampling manifests.
   - resumable processing using JSONL manifest.
4. **Validation layer**
   - synthetic detector sanity tests.
   - optional labeled evaluation helper if labels exist.


In [ ]:
# Install dependencies if needed:
# %pip install -q opencv-python pandas tqdm

from __future__ import annotations

from pathlib import Path

from streetview_filter import (
    FilterConfig,
    StreetViewInvalidFilterPipeline,
)
from streetview_filter.utils import compute_dhash, compute_phash
from streetview_filter.tests import run_detector_self_tests


In [ ]:
# ----------------------------
# Configuration
# ----------------------------
# All configuration dataclasses (ThresholdConfig, WeightConfig, RuntimeConfig,
# FilterConfig) are now defined in streetview_filter/config.py.
# Import and customise as needed:
#
#   from streetview_filter.config import FilterConfig, ThresholdConfig
#   config = FilterConfig(thresholds=ThresholdConfig(dark_mean_v=40.0))
#
# The default config is also available as:
#   from streetview_filter import DEFAULT_CONFIG


In [ ]:
# ----------------------------
# User inputs / run settings
# ----------------------------

# Optional Colab mount
try:
    from google.colab import drive  # type: ignore
except Exception:
    drive = None

# if drive is not None:
#     drive.mount('/content/drive')

INPUT_FOLDER = Path("/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/Test_Images")
OUTPUT_FOLDER = Path("/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/Filter_Output_v3")

CONFIG = FilterConfig(
    placeholder_reference_paths=[
        "/content/drive/MyDrive/1-UCL_Classes/Advanced_SGDS/Preservation_Final_Project/Images/invalid_image.jpg"
    ],
    logo_template_paths=[
        # Add template paths if you have them, e.g.:
        # "/content/drive/MyDrive/.../google_logo_template.png"
    ],
)

# Optional targeted overrides for your first rerun
# CONFIG.thresholds.building_min_score = 0.33
# CONFIG.thresholds.invalid_score_threshold = 0.64
# CONFIG.runtime.invalid_action = "copy"

print("Input exists:", INPUT_FOLDER.exists())
print("Output exists:", OUTPUT_FOLDER.exists())
print("Placeholder refs:", CONFIG.placeholder_reference_paths)


In [ ]:
# ----------------------------
# Shared helpers
# ----------------------------
# All shared helper functions (clamp, normalize, inverse_normalize,
# weighted_sum, load_grayscale_images, morphology_open_bool,
# logo_absence_score, sample_from_dataframe, utc_now_iso, etc.)
# are now centralised in streetview_filter/utils.py.
#
# Image-hashing (compute_dhash, compute_phash, hash_similarity,
# corr_similarity), debug overlay, and image-discovery functions
# are also in that module.
print("Shared helpers loaded from streetview_filter.utils")


In [ ]:
# ----------------------------
# Multi-cue detectors
# ----------------------------
# The MultiCueDetectors class is now in streetview_filter/detectors.py.
# Key refactorings:
#   - _load_placeholder_refs / _load_logo_templates now share
#     load_grayscale_images() from utils.py
#   - _compute_masks uses morphology_open_bool() instead of
#     3x inline cv2.morphologyEx calls
#   - All weighted-score computations use weighted_sum()
#   - _score_non_google / _score_interior share logo_absence_score()
#   - analyze() extracts logo_available once instead of computing
#     bool(scores['logo_templates_available'] > 0.5) twice
print("MultiCueDetectors loaded from streetview_filter.detectors")


In [ ]:
# ----------------------------
# Decision policy + pipeline
# ----------------------------
# DecisionEngine is now in streetview_filter/decision.py.
# StreetViewInvalidFilterPipeline and evaluate_with_labels are in
# streetview_filter/pipeline.py.
# Key refactorings:
#   - generate_review_samples uses sample_from_dataframe() to
#     eliminate 3x duplicated sampling logic
#   - Timestamps use utc_now_iso() instead of inline
#     dt.datetime.utcnow().isoformat() + 'Z'
print("DecisionEngine loaded from streetview_filter.decision")
print("StreetViewInvalidFilterPipeline loaded from streetview_filter.pipeline")


In [ ]:
# ----------------------------
# STEP 3 - Entry point: run the pipeline
# ----------------------------

pipeline = StreetViewInvalidFilterPipeline(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    config=CONFIG,
)

# Set max_images=None for full run (recommended for production pass)
# For quick smoke test, set max_images=200
results_df = pipeline.run(max_images=None)

print("\nDone.")
print("Total images in manifest:", len(results_df))
if len(results_df):
    print("Invalid count:", int(results_df["is_invalid"].sum()))
    print("Top primary reasons:")
    print(results_df["primary_reason"].value_counts().head(10))

print("\nOutputs:")
print("-", pipeline.results_csv_path)
print("-", pipeline.results_jsonl_path)
print("-", pipeline.results_json_path)
print("-", pipeline.summary_json_path)
print("-", pipeline.review_dir / "review_manifest_v3.csv")


In [ ]:
# ----------------------------
# STEP 3b - Lightweight detector self-tests
# ----------------------------
# Tests are now in streetview_filter/tests.py

# Uncomment to run sanity tests:
# run_detector_self_tests()


## STEP 6 - Threshold tuning quick guide (first 20k rerun)

1. Start with `invalid_action='copy'` and inspect `reports/review_manifest_v3.csv` plus debug overlays.
2. If too many valid facades are rejected for low-building/open-scene:
   - decrease `thresholds.building_min_score` by ~0.02
   - increase `thresholds.invalid_score_threshold` by ~0.02
3. If obvious corridor/no-facade images are slipping through:
   - decrease `thresholds.road_corridor`
   - increase `weights.road_corridor` slightly (e.g., +0.02)
4. If blur misses obvious blurry buildings:
   - decrease `thresholds.blur_soft_severity`
   - keep `blur_hard_severity` conservative to avoid overfiring
5. For placeholder/no-imagery misses:
   - add more reference placeholders to `placeholder_reference_paths`
   - lower `placeholder_soft_similarity` modestly (e.g., 0.88 -> 0.85)
6. Keep `non_google` as a weak/demoted signal unless you have strong templates and evidence.
